In [9]:
# import numpy as np
# import matplotlib.pyplot as plt

# from lammps import lammps, LMP_STYLE_GLOBAL,LMP_STYLE_ATOM, LMP_STYLE_LOCAL, LMP_TYPE_SCALAR
# from lammps import LMP_TYPE_VECTOR, LMP_TYPE_ARRAY, LMP_SIZE_VECTOR, LMP_SIZE_ROWS, LMP_SIZE_COLS
# from ctypes import c_double, c_int

# import scipy
# from scipy import optimize

In [1]:
import LACT
from LACT import *

In [2]:
import numpy as np
import matplotlib.pyplot as plt

from lammps import lammps, LMP_STYLE_ATOM, LMP_TYPE_VECTOR, LMP_TYPE_ARRAY

#from lammps import lammps, LMP_STYLE_GLOBAL,LMP_STYLE_ATOM, LMP_STYLE_LOCAL, LMP_TYPE_SCALAR
#from lammps import LMP_TYPE_VECTOR, LMP_TYPE_ARRAY, LMP_SIZE_VECTOR, LMP_SIZE_ROWS, LMP_SIZE_COLS
from ctypes import c_double, c_int

In [3]:
#lmp = lammps()
lmp = lammps(cmdargs=['-screen', 'none'])

lmp.commands_string('''
# ------------------------ INITIALIZATION ----------------------------
processors    * * *
units         metal
dimension    3
boundary    p    p    p
atom_style   atomic
atom_modify map yes


box        tilt large
#--------------------------- LAMMPS Data File--------------------------
#read_data    input_data/thicker_domain.lmp
read_data    input_data/4b_step.lmp
change_box    all triclinic
reset_atom_ids sort yes

#--------------
pair_style    eam/alloy
pair_coeff    * * input_data/Cu_mishin1.eam.alloy Cu

#-------------------Various continuation commands----------------------
compute forces all property/atom fx fy fz
compute ids all property/atom id
compute x_check all property/atom x y z
#atom_modify map yes
''')

natoms = lmp.extract_global("natoms")
original_box_size = lmp.extract_box()
ref_X = np.reshape(
            np.array(lmp.gather_atoms("x", 1, 3)),
            (natoms, 3),
        ).copy()

In [4]:
# #### helper functions

# ### periodicity preservation
# def fix_periodicity(X,box_size,show=False):
#     kk = 0
#     for ii in range(len(X)):
#         for j in range(3):
#             if X[ii][j] < box_size[0][j]:
#                 X[ii][j] += box_size[1][j] - box_size[0][j]
#                 kk+=1
#             elif X[ii][j] > box_size[1][j]:
#                 X[ii][0] -= box_size[1][j] - box_size[0][j]
#                 kk+=1
#     if show:
#         print("Number of changed coordinates:",kk)
    
# def fix_periodicity_flat(X,box_size,show=False):
#     kk = 0
#     for ii in range(len(X)):
#         j = ii % 3
#         if X[ii] < box_size[0][j]:
#             X[ii] += box_size[1][j] - box_size[0][j]
#             kk+=1
#         elif X[ii] > box_size[1][j]:
#             X[ii] -= box_size[1][j] - box_size[0][j]
#             kk+=1
#     if show:
#         print("Number of changed coordinates:",kk)
        
# def fix_periodicity_relative(X,box_size,show=False):
#     kk = 0
#     allowed_directions = (np.array(box_size[1]) - np.array(box_size[0]))/2
#     for ii in range(len(X)):
#         for j in range(3):
#             if X[ii][j] < -allowed_directions[j]:
#                 X[ii][j] += 2*allowed_directions[j]
#                 kk+=1
#             elif X[ii][j] > allowed_directions[j]:
#                 X[ii][0] -= 2*allowed_directions[j]
#                 kk+=1
#     if show:
#         print("Number of changed coordinates:",kk)
        
# def fix_periodicity_relative_flat(X,box_size,show=False):
#     kk = 0
#     allowed_directions = (np.array(box_size[1]) - np.array(box_size[0]))/2
#     for ii in range(len(X)):
#         j = ii % 3
#         if X[ii] < -allowed_directions[j]:
#             X[ii] += 2*allowed_directions[j]
#             kk+=1
#         elif X[ii] > allowed_directions[j]:
#             X[ii] -= 2*allowed_directions[j]
#             kk+=1
        
#     if show:
#         print("Number of changed coordinates:",kk)
        
# ### LAMMPS interface:

# # passing the flat dN+1 variable Y to LAMMPS
# def pass_X(Y):
#     lammps_commands = f"""
#      change_box all x final {original_box_size[0][0]+Y[-1]} {original_box_size[1][0]-Y[-1]} units box
#     #fix        relax all box/relax iso 0.0    
#     """
#     lmp.commands_string(lammps_commands)
#     box_size = lmp.extract_box()[:2]
#     Y_x = Y[0:-1]
#     fix_periodicity_flat(Y_x,box_size)
#     Y_x = ((len(Y_x))*c_double)(*Y_x)
#     lmp.scatter_atoms("x", 1, 3, Y_x)

# # extended system
# def G(Y,Y0,Ydot,ds,verbose = False):
#     pass_X(Y)
#     lmp.command('run 0')
#     f_t = lmp.numpy.extract_compute('forces',LMP_STYLE_ATOM,LMP_TYPE_ARRAY)
#     _IDS = lmp.numpy.extract_compute('ids',LMP_STYLE_ATOM,LMP_TYPE_VECTOR).astype('int32')
#     f_t = f_t[np.argsort(_IDS)]
#     G0 = f_t.flatten()
#     box_size = lmp.extract_box()[:2]
#     YminusY0 = Y-Y0
#     fix_periodicity_relative_flat(YminusY0[:-1],box_size)
#     last_eqn = (YminusY0*Ydot).sum() - ds
#     G0 = np.append(G0,last_eqn)
#     if verbose:
#         print(Y[-1])
#         print(last_eqn)
#         print(np.linalg.norm(G0,np.Inf))
#         #lmp.command('write_dump all custom run_dump2 id type x y z xu yu zu')
#     return G0

In [5]:
#energies
E = []
#optimal extended variables
Y = []


# Do we start it as new continuation run or continue one from before?
new_start = True

if new_start:
    #populate them with two consecutive data points from previous runs:
    ii = 44
    forward = True
    rr = range(2) if forward else reversed(range(1))

    for k in rr:
        lmp.command(f'read_dump dumps/thin_domain/minimise_dump/minimise_dump_{ii+k} 0 x y z box yes')
        lmp.command('run 0')
        #_E = lmp.get_thermo("pe")
        #_max_force = lmp.get_thermo("fmax")
        #_IDS = np.array(lmp.gather_atoms("id", 0, 1))
        #_IDS = lmp.numpy.extract_compute('ids',LMP_STYLE_ATOM,LMP_TYPE_VECTOR).astype('int32')
        #_F = lmp.numpy.extract_compute('forces',LMP_STYLE_ATOM,LMP_TYPE_ARRAY).copy()
        _X = np.reshape(
                 np.array(lmp.gather_atoms("x", 1, 3)),
                 (natoms, 3),
             )
        _μ = -(original_box_size[0][0] - lmp.extract_box()[0][0])
        _Y= np.append(_X.flatten(),_μ)
        Y += [_Y]
        lmp.command(f'write_dump all custom dumps/thin_domain/continuation_dumps/default_branch/dump_{len(Y)} id type x y z xu yu zu')  
        
else:
    for k in range(1348): 
        lmp.command(f'read_dump dumps/thin_domain/continuation_dumps/default_branch/dump_{k+1} 0 x y z box yes')
        lmp.command('run 0')
        _X = np.reshape(
                 np.array(lmp.gather_atoms("x", 1, 3)),
                 (natoms, 3),
             )
        _μ = -(original_box_size[0][0] - lmp.extract_box()[0][0])
        _Y= np.append(_X.flatten(),_μ)
        Y += [_Y]
        lmp.command('reset_timestep 0')
        


 

In [6]:
len(Y)

2

In [7]:
def continuation_step(Y,ds):
    #print(len(Y))
    pass_X(Y[-1])
    box_size = lmp.extract_box()[:2]
    Y_dot = Y[-1] - Y[-2]
    fix_periodicity_relative_flat(Y_dot[:-1],box_size)
    Y_dot = Y_dot / np.linalg.norm(Y_dot)    
    Y_0 = Y[-1] + ds*Y_dot
    
    Y_1 = scipy.optimize.root(G, Y_0,
                   args=(Y[-1],Y_dot,ds),
#                    tol = 1e-4,
                   method='krylov',
                   options={'disp': True,
                            'fatol': 1e-4,
                            'maxiter': 6,
                            'line_search': 'armijo' ,
                            'jac_options': {'inner_M': None, 
                                            'method': 'lgmres'}},
                   callback=None)
    return Y_1

In [8]:
ds_default = 1e-2
ds_smallest = 1e-3
ds_largest = 2e0
ds = ds_default
counter = 0

In [9]:
for k in range(300):
    Y_1 = continuation_step(Y,ds)
    if Y_1.success == False:
        ds = ds/2
        if ds > ds_smallest:
            print("ds halved, now equal to ", ds, ". We go 2 steps back.")
        else:
            print("ds now below the threshold, equal to ", ds, ".  Aborting...")
            break
        res_at = -2
        Y = Y[:res_at]
        counter = 0
    else:
        Y += [Y_1.x]
        pass_X(Y[-1])
        lmp.command('reset_timestep 0')
        lmp.command(f'write_dump all custom dumps/thin_domain/continuation_dumps/default_branch/dump_{len(Y)} id type x y z xu yu zu')        
        counter += 1
        if counter > 5 and ds < ds_largest:
            ds = 2*ds
            print("ds doubled, now equal to ", ds, ".")
            counter = 0
        
        print("Iteration step: ",k," ",", Solution step: ",len(Y)," ",", Continuation parameter: ", Y[-1][-1])
    if np.sign(Y[-1][-1] - Y[-2][-1])*np.sign(Y[-2][-1] - Y[-3][-1]) < 0.0:
        print("turn, turn, turn")
    

NameError: name 'original_box_size' is not defined

In [19]:
ii = -2

pass_X(Y[ii-2])
box_size = lmp.extract_box()[:2]
Y_dot = Y[ii-2] - Y[ii-3]
fix_periodicity_relative_flat(Y_dot[:-1],box_size)
Y_dot = Y_dot / np.linalg.norm(Y_dot)

np.linalg.norm(G(Y[ii-1],Y[ii-2],Y_dot,0,verbose=True)[:-1],np.Inf)

4.659600176939321
0.039997306659327325
0.039997306659327325


8.829545681526829e-05

In [18]:
len(Y)

2272

In [21]:
Y[-4][-2]

472.54634247754325

In [24]:
Y[-5]

array([-59.37911416,   3.33501287, 401.64844665, ...,   1.85414103,
       472.54600213,   4.65763705])